# 01 - Fundamentos de Agrupamiento con K-Means (Algoritmo de Lloyd)

## Curso de Machine Learning: Modelos No Supervisados

En este cuaderno iniciaremos el estudio del **Aprendizaje No Supervisado** abordando el algoritmo de particionamiento mas relevante y utilizado en la industria: **K-Means**. A diferencia de los modelos supervisados de regresion y clasificacion estudiados previamente, donde buscabamos aprender un mapeo $f: X \rightarrow y$ guiados por una variable objetivo conocida, en el agrupamiento (*clustering*) trabajamos unicamente con la matriz de caracteristicas $X \in \mathbb{R}^{n \times p}$ sin etiquetas preexistentes.

El objetivo fundamental de K-Means es descubrir estructuras latentes en los datos dividiendo las $n$ observaciones en $K$ grupos disjuntos (clusters) $C_1, C_2, \dots, C_K$, de tal manera que las observaciones dentro de un mismo cluster sean altamente similares entre si (alta cohesion interna) y muy distintas a las observaciones de otros clusters (alta separacion externa).

---

### Contenidos
1. Motivacion: Del Aprendizaje Supervisado al Descubrimiento de Estructuras Latentes
2. Intuicion Geometrica: Teselacion de Voronoi y Particionamiento del Espacio
3. Formulacion Matematica: Minimización de la Varianza Intra-Cluster (WCSS / Inercia)
4. El Algoritmo de Lloyd (Expectation-Maximization Discreto)
5. Sensibilidad a la Escala y Necesidad Imperativa de Estandarizacion
6. Limitaciones Topologicas de K-Means: Esfericidad, Isotropia y Densidad
7. Caso de Estudio Real:Quimiometria e Identificacion de Perfiles Enologicos (Dataset Wine)
8. Visualizacion de Fronteras de Decision en 2D
9. Comparativa: K-Means sin Escalar vs. K-Means Estandarizado
10. Conclusiones


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_wine, make_blobs, make_moons
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.pipeline import Pipeline
from sklearn.metrics import silhouette_score, adjusted_rand_score

# Configuracion visual
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.random.seed(42)


## 1. Motivacion: Del Aprendizaje Supervisado al Descubrimiento de Estructuras Latentes

En la practica profesional de Ciencia de Datos e Ingenieria de Datos, obtener etiquetas verificadas ($y$) suele ser un proceso costoso, lento o directamente imposible a gran escala. Sin embargo, los datos crudos ($X$) abundan.

El agrupamiento mediante **K-Means** resuelve problemas criticos de negocio e ingenieria donde no existe una verdad fundamental a priori:
- **Segmentacion estrategica de clientes o activos**: Descubrir arquetipos de comportamiento para disenar politicas diferenciadas.
- **Compresion de datos y Cuantizacion Vectorial**: Reducir el espacio de estados representando millones de registros continuos mediante un conjunto discreto de $K$ prototipos (centroides).
- **Preprocesamiento e Ingenieria de Caracteristicas**: Generar nuevas variables categoricas (el ID del cluster asignado o la distancia a cada centroide) para alimentar modelos supervisados posteriores.
- **Deteccion de anomalias**: Identificar observaciones que se encuentran a una distancia extrema del centroide mas cercano.

A diferencia de los Arboles de Decision (CART), que son invariantes a transformaciones monotonicas de las variables porque operan con umbrales ordinales ($x_j \leq t$), **K-Means es un algoritmo puramente geometrico basado en distancias metricas**, lo cual impone requisitos estrictos sobre el preprocesamiento de los datos.


## 2. Intuicion Geometrica: Teselacion de Voronoi

Geometricamente, K-Means define cada cluster $k \in \{1, \dots, K\}$ mediante un punto representativo en el espacio de caracteristicas $\mathbb{R}^p$ denominado **centroide** ($\mu_k$), el cual corresponde al centro de masa (media aritmetica multivariada) de los puntos asignados a dicho grupo.

Una vez fijados los $K$ centroides $\{\mu_1, \mu_2, \dots, \mu_K\}$, el espacio $\mathbb{R}^p$ queda particionado en $K$ regiones convexas conocidas como **Celdas de Voronoi**:

$$C_k = \{ x_i \in X : \|x_i - \mu_k\|_2^2 \leq \|x_i - \mu_j\|_2^2 \quad \forall j \neq k \}$$

Las fronteras que separan cualquier par de clusters adyacentes son **hiperplanos perpendiculares al segmento que une sus respectivos centroides** (bisectrices perpendiculares). A diferencia de CART, donde las fronteras eran obligatoriamente paralelas a los ejes coordenados, K-Means genera fronteras lineales con cualquier orientacion angular en el espacio.


## 3. Formulacion Matematica: Minimización de la Inercia (WCSS)

Formalmente, K-Means busca encontrar la particion $C = \{C_1, \dots, C_K\}$ y los centroides $\{\mu_1, \dots, \mu_K\}$ que minimicen la **Suma de Cuadrados Intra-Cluster** (*Within-Cluster Sum of Squares*, WCSS), denominada en Scikit-Learn como **Inercia**:

$$J(C, \mu) = \sum_{k=1}^{K} \sum_{x_i \in C_k} \|x_i - \mu_k\|_2^2 = \sum_{k=1}^{K} \sum_{x_i \in C_k} \sum_{j=1}^{p} (x_{ij} - \mu_{kj})^2$$

Donde:
- $x_i \in \mathbb{R}^p$ es el vector de caracteristicas de la observacion $i$.
- $\mu_k = \frac{1}{|C_k|} \sum_{x_i \in C_k} x_i$ es el vector centroide del cluster $C_k$.
- $\|x_i - \mu_k\|_2^2$ es la distancia euclidiana al cuadrado entre el punto $x_i$ y su centroide asignado $\mu_k$.

**Complejidad Computacional**: Encontrar el minimo global exacto de la funcion objetivo $J(C, \mu)$ es un problema **NP-Duro** incluso para $K=2$ en un espacio general $\mathbb{R}^p$. Por ello, recurrimos a heuristicas iterativas que convergen a minimos locales.


## 4. El Algoritmo de Lloyd (Iteracion en Dos Pasos)

El algoritmo estandar para resolver K-Means fue propuesto por Stuart Lloyd (1957) y opera mediante una estrategia de optimizacion alternada estrechamente relacionada con el algoritmo *Expectation-Maximization* (EM):

### Paso 0: Inicializacion
Se seleccionan $K$ vectores iniciales $\mu_1^{(0)}, \dots, \mu_K^{(0)} \in \mathbb{R}^p$ (de forma aleatoria o mediante muestreo ponderado inteligente **K-Means++**, que es el estandar en Scikit-Learn).

### Paso 1: Asignacion de Clusters (Paso E - Expectation)
Manteniendo fijos los centroides $\mu_k^{(t)}$, se asigna cada observacion $x_i$ al centroide mas cercano segun la distancia euclidiana:
$$c_i^{(t)} = \arg\min_{k \in \{1,\dots,K\}} \|x_i - \mu_k^{(t)}\|_2^2$$

### Paso 2: Actualizacion de Centroides (Paso M - Maximization)
Manteniendo fijas las asignaciones $C_k^{(t)}$, se recalcula la posicion de cada centroide como el promedio exacto de los puntos asignados a el (dado que la media aritmetica minimiza el error cuadratico):
$$\mu_k^{(t+1)} = \frac{1}{|C_k^{(t)}|} \sum_{x_i \in C_k^{(t)}} x_i$$

### Convergencia
Los Pasos 1 y 2 se repiten iterativamente hasta que las asignaciones no cambien o el desplazamiento de los centroides sea menor a una tolerancia predefinida ($\|\mu_k^{(t+1)} - \mu_k^{(t)}\|_2 < \text{tol}$). Como cada paso reduce monotonamente la funcion de costo $J$ y existe un numero finito de particiones ($K^n$), **la convergencia a un optimo local esta matematicamente garantizada**.


## 5. Demostracion Visual: Teselacion de Voronoi y Limitaciones Topologicas

Antes de analizar datos quimiometricos reales, visualicemos en 2D como K-Means particiona el espacio mediante celdas de Voronoi y que ocurre cuando se violan sus supuestos geometricos fundamentales (clusters no esfericos o no convexos).


In [ ]:
# Generacion de dos escenarios sinteticos: Blobs Isotropicos vs Lunas No Convexas
X_blobs, _ = make_blobs(n_samples=400, centers=4, cluster_std=0.85, random_state=42)
X_moons, _ = make_moons(n_samples=400, noise=0.07, random_state=42)

km_blobs = KMeans(n_clusters=4, n_init=10, random_state=42).fit(X_blobs)
km_moons = KMeans(n_clusters=2, n_init=10, random_state=42).fit(X_moons)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Funcion auxiliar para graficar fronteras de Voronoi
def plot_voronoi_regions(ax, model, X, title):
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 400), np.linspace(y_min, y_max, 400))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, cmap='Pastel1', alpha=0.5)
    ax.contour(xx, yy, Z, colors='k', linewidths=0.8, linestyles='--')
    ax.scatter(X[:, 0], X[:, 1], c=model.labels_, cmap='Set1', s=30, edgecolor='k', alpha=0.8)
    ax.scatter(model.cluster_centers_[:, 0], model.cluster_centers_[:, 1], 
               c='black', marker='X', s=220, edgecolor='white', linewidth=2, label='Centroides ($\mu_k$)')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('$x_1$', fontsize=10)
    ax.set_ylabel('$x_2$', fontsize=10)
    ax.legend(loc='upper right', fontsize=10)

plot_voronoi_regions(axes[0], km_blobs, X_blobs, 'Caso Ideal: Clusters Esfericos (Teselacion de Voronoi)')
plot_voronoi_regions(axes[1], km_moons, X_moons, 'Limitacion Geometrica: Formas No Convexas (Frontera Lineal)')

plt.suptitle('Geometria de Particionamiento en K-Means', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


## 6. Terminologia Formal del Agrupamiento con K-Means

| Termino | Definicion |
| :--- | :--- |
| **Centroide ($\mu_k$)** | Vector de medias aritmeticas de todas las observaciones pertenecientes al cluster $C_k$. Actua como el prototipo del grupo. |
| **Inercia (WCSS)** | Suma de las distancias euclidianas al cuadrado desde cada punto hasta su centroide asignado. Mide la compacidad interna. |
| **K-Means++** | Algoritmo de inicializacion probabilistica que selecciona los centroides iniciales alejados entre si para acelerar la convergencia y evitar malos minimos locales. |
| **Epoca / Iteracion** | Un ciclo completo de asignacion de todos los puntos a su centroide mas cercano seguido del recalculo de la posicion de los centroides. |
| **`n_init`** | Numero de veces que el algoritmo de K-Means se ejecuta de forma independiente con distintas semillas de centroides iniciales, conservando el modelo con menor inercia final. |
| **Celda de Voronoi** | Region geometrica convexa del espacio $\mathbb{R}^p$ que contiene todos los puntos mas cercanos a $\mu_k$ que a cualquier otro centroide $\mu_j$. |


## 7. Caso de Estudio Real: Quimiometria Enologica (Dataset Wine)

A lo largo de los cuadernos teoricos trabajaremos con un conjunto de datos clasico de **Quimiometria Analitica**: el perfil fisico-quimico de vinos producidos en la region del Piamonte (Italia) a partir de tres cepas distintas (Barolo, Grignolino y Barbera).

Este conjunto de datos contiene **13 variables continuas** que miden concentraciones de metabolitos secundarios, fenoles, flavonoides, minerales (magnesio, alcalinidad de cenizas), intensidad cromatica y prolina.

**Importante**: En nuestro flujo de entrenamiento **eliminaremos por completo la etiqueta de la cepa** (`target`). Nuestro objetivo es comprobar si K-Means es capaz de descubrir de forma **100% ciega y no supervisada** la existencia de estas tres familias quimicas basandose exclusivamente en sus perfiles espectrofotometricos, y por que el escalado previo es el factor determinante entre el exito rotundo y el fracaso absoluto.


In [ ]:
# Carga del dataset quimiometrico
wine = load_wine(as_frame=True)
X_wine = wine.data
y_true_cepas = wine.target  # Exclusivamente para auditoria externa posterior, NUNCA entra al modelo

print(f"Dimensiones de la matriz quimiometrica X: {X_wine.shape}")
print("\nEstadisticas descriptivas de las variables (observese la disparidad extrema de escalas):")
print(X_wine.describe().T[['mean', 'std', 'min', 'max']].round(2))


## 8. El Efecto Critico de la Escala: K-Means Crudo vs. K-Means Estandarizado

Observa la tabla anterior: mientras que `nonflavanoid_phenols` tiene una desviacion estandar de $\approx 0.12$, el aminoacido `proline` tiene una desviacion estandar de $\approx 314.91$ (mas de 2,500 veces mayor).

Al calcular la distancia euclidiana al cuadrado entre dos vinos $x_a$ y $x_b$:
$$\|x_a - x_b\|_2^2 = \sum_{j=1}^{13} (x_{aj} - x_{bj})^2$$

La diferencia cuadratica en `proline` sera del orden de $(300)^2 = 90,000$, mientras que en `nonflavanoid_phenols` sera de $(0.1)^2 = 0.01$. **Si no estandarizamos los datos con `StandardScaler`, K-Means ignorara 12 de las 13 variables quimicas y agrupara los vinos exclusivamente por su nivel de prolina y magnesio.**

Comprobemos este fenomeno empiricamente entrenando ambos modelos.


In [ ]:
# 1. Modelo K-Means sobre datos CRUDOS (sin estandarizar)
kmeans_raw = KMeans(n_clusters=3, init='k-means++', n_init=10, random_state=42)
labels_raw = kmeans_raw.fit_predict(X_wine)

# 2. Pipeline Profesional: StandardScaler + K-Means
scaler = StandardScaler()
X_wine_scaled = scaler.fit_transform(X_wine)

kmeans_scaled = KMeans(n_clusters=3, init='k-means++', n_init=10, random_state=42)
labels_scaled = kmeans_scaled.fit_predict(X_wine_scaled)

# Evaluacion mediante metricas internas (Silhouette) y auditoria externa (Adjusted Rand Index)
sil_raw = silhouette_score(X_wine, labels_raw)
ari_raw = adjusted_rand_score(y_true_cepas, labels_raw)

sil_scaled = silhouette_score(X_wine_scaled, labels_scaled)
ari_scaled = adjusted_rand_score(y_true_cepas, labels_scaled)

print("=" * 75)
print("IMPACTO DE LA ESTANDARIZACION EN K-MEANS (K=3)")
print("=" * 75)
print(f"K-Means Sin Escalar:    Inercia = {kmeans_raw.inertia_:10.2f} | Silhouette = {sil_raw:.4f} | Pureza (ARI) = {ari_raw:.4f}")
print(f"K-Means Estandarizado:  Inercia = {kmeans_scaled.inertia_:10.2f} | Silhouette = {sil_scaled:.4f} | Pureza (ARI) = {ari_scaled:.4f}")
print("=" * 75)


## 9. Visualizacion Comparativa del Particionamiento

El **Adjusted Rand Index (ARI)** mide la concordancia entre los clusters descubiertos a ciegas y las cepas reales (donde 0.0 equivale a asignacion aleatoria y 1.0 a coincidencia perfecta). Observa como el ARI salta de un mediocre **0.3711** a un extraordinario **0.8975** simplemente al aplicar `StandardScaler`.

Visualicemos ambos particionamientos proyectados sobre dos variables quimicas fundamentales: `flavanoids` y `color_intensity`.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

feat_x, feat_y = 'flavanoids', 'color_intensity'

# Panel 1: K-Means Sin Escalar
axes[0].scatter(X_wine[feat_x], X_wine[feat_y], c=labels_raw, cmap='viridis', s=50, edgecolor='k', alpha=0.8)
axes[0].set_title(f'K-Means Sin Escalar\n(ARI = {ari_raw:.3f} - Mezcla Severa)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Flavonoides (mg/L)', fontsize=10)
axes[0].set_ylabel('Intensidad de Color', fontsize=10)

# Panel 2: K-Means Estandarizado
axes[1].scatter(X_wine[feat_x], X_wine[feat_y], c=labels_scaled, cmap='viridis', s=50, edgecolor='k', alpha=0.8)
axes[1].set_title(f'K-Means Estandarizado (StandardScaler)\n(ARI = {ari_scaled:.3f} - Separacion Nitida)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Flavonoides (mg/L)', fontsize=10)
axes[1].set_ylabel('Intensidad de Color', fontsize=10)

# Panel 3: Cepas Reales (Referencia Oculta)
axes[2].scatter(X_wine[feat_x], X_wine[feat_y], c=y_true_cepas, cmap='Set1', s=50, edgecolor='k', alpha=0.8)
axes[2].set_title('Referencia Quimiometrica Real\n(3 Cepas Verdaderas)', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Flavonoides (mg/L)', fontsize=10)
axes[2].set_ylabel('Intensidad de Color', fontsize=10)

plt.suptitle('Descubrimiento Ciego de Perfiles Quimicos: Efecto de StandardScaler', fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()


## 10. Conclusiones

### Puntos Clave:
1. **Naturaleza Geometrica**: K-Means particiona el espacio en celdas de Voronoi convexas minimizando la varianza intra-cluster (Inercia / WCSS).
2. **Estandarizacion Obligatoria**: Debido al uso de la distancia euclidiana isotropicamente en todas las dimensiones, variables con mayor varianza nominal dominan por completo la funcion de costo. El uso de `StandardScaler` (o `RobustScaler` en presencia de valores atipicos moderados) es un requisito innegociable.
3. **Supuestos Estructurales**: K-Means asume implicitamente que los clusters subyacentes son convexos, aproximadamente esfericos (misma varianza en todas las direcciones) y de tamanos comparables.
4. **El Hiperparametro Critico ($K$)**: En este cuaderno fijamos $K=3$ porque conociamos de antemano el contexto enologico. Sin embargo, en un problema real de aprendizaje no supervisado, **$K$ es desconocido**. 

En el siguiente cuaderno, **02_Seleccion_Optima_de_K_y_Metricas_Evaluacion.ipynb**, estudiaremos rigurosamente las metricas matematicas internas (Silueta, Calinski-Harabasz, Davies-Bouldin) y las variantes algoritmicas para determinar el numero optimo de clusters cuando no existe ninguna etiqueta de referencia.
